### Algorithms for Massive Datasets Project Source Code
- Author: Andrea Colombo

In [1]:
# Kaggle setup for downloading the dataset, this has to be handled before
# running the rest of the notebook to avoid errors
import os

os.environ["KAGGLE_USERNAME"] = "x"
os.environ["KAGGLE_KEY"] = "x"

In [2]:
# Generic configuration related to the actual project implementation
# The user can change how much of the dataset is actually used during the run
# and other global variables that influence the behaviour of the notebook

ENABLE_LOGGING = True
SAMPLING_PROPORTION = 1.0 # This can range from 0.0 (excluded) to 1.0
assert 0 < SAMPLING_PROPORTION <= 1, "Invalid sampling factor"
RAND_SEED = 42 # Set as None for random, used for dataset sampling
FM_NUM_HASHES = 256 # Number of hash function used for the FM algorithm
FM_GROUP_SIZES = 4
assert FM_NUM_HASHES % FM_GROUP_SIZES == 0
FM_PHI = 0.77351
FM_USE_KM_DERIVATION = False # Derive FN_NUM_HASHES from 2 hashes

def log(s):
    if ENABLE_LOGGING == True:
        print(f"[LOG] {s}")


In [3]:
# Get the dataset using kaggle API (please check the kaggle auth cell before running
# this one)

%pip install kaggle
%pip install xxhash
!kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020"
!unzip -o -d dataset new-york-times-articles-comments-2020.zip && rm -r new-york-times-articles-comments-2020.zip
# !ls -l dataset

Dataset URL: https://www.kaggle.com/datasets/benjaminawd/new-york-times-articles-comments-2020
License(s): CC-BY-NC-SA-4.0
100% 1.95G/1.95G [00:29<00:00, 71.4MB/s]

Archive:  new-york-times-articles-comments-2020.zip
  inflating: dataset/nyt-articles-2020.csv  
  inflating: dataset/nyt-comments-2020.csv  
  inflating: dataset/nyt-comments-part0.csv  
  inflating: dataset/nyt-comments-part1.csv  
  inflating: dataset/nyt-comments-part2.csv  
  inflating: dataset/nyt-comments-part3.csv  
  inflating: dataset/nyt-comments-part4.csv  
  inflating: dataset/nyt-comments-part5.csv  
  inflating: dataset/nyt-comments-part6.csv  
  inflating: dataset/nyt-comments-part7.csv  
  inflating: dataset/nyt-comments-part8.csv  
  inflating: dataset/nyt-comments-part9.csv  
  inflating: dataset/test.csv        
  inflating: dataset/train.csv       


In [ ]:
%pip install pyspark
%pip install py4j

import pyspark

log(f"PySpark Ver: {pyspark.__version__}")

spark_session = pyspark.sql.SparkSession.builder.getOrCreate()
spark_context = spark_session.sparkContext

In [ ]:
df = (spark_session.read
      .option("header", "true")
      .option("inferSchema", "true")
      .option("quote", '"')
      .option("escape", '"')
      .option("multiLine", "true")
      .option("mode", "PERMISSIVE")
      .csv("dataset/nyt-comments-2020.csv")
      .sample(withReplacement=False, fraction=SAMPLING_PROPORTION, seed=RAND_SEED)
      .cache())

# df.show(5, truncate=False)
# df.printSchema()
log(f"Dataset columns {df.columns}")
# log(f"Dataset count {df.count()}")

In [ ]:
import os

# Set up user streams for partitions
# The dataset has null entries so they have to be cleaned up to avoid
# null strings giving us garbage with the hashes
users_stream = (
    df.
    select("userID").
    where("userID IS NOT NULL").
    rdd.
    map(lambda row: row["userID"])
)

# TODO: check why multiline csv parsing forces it to one partition
# just use default values (same as cpu cores?) for now
users_stream = users_stream.repartition(spark_context.defaultParallelism)
users_stream.getNumPartitions()

# FM Algorithm Implementation

Each userID is used to calculate _n_ hash functions (FM_NUM_HASHES). We keep track of the
maximum number of trailing zeros for each has function and we use the registers of each
partition for the final estimate

In [ ]:
import xxhash
import statistics
from random import randint

def ctz(x):
    if x == 0:
        return 64
    return (x & -x).bit_length() - 1

def hash64bits(value, seed):
    return xxhash.xxh3_64(value.to_bytes(8, 'little'), seed).intdigest()

def fm_partition(it):
    registers = [0] * FM_NUM_HASHES
    for user in it:
        for i in range(FM_NUM_HASHES):
            bits = hash64bits(user, i)
            r = ctz(bits)
            if r > registers[i]:
                registers[i] = r

    yield registers


In [ ]:
mapped = users_stream.mapPartitions(fm_partition)
registers = mapped.reduce(lambda a, b: [max(x, y) for x, y in zip(a, b)])

In [ ]:
# Naive with no groups
estimates = [(2 ** r) / FM_PHI for r in registers]
fm_estimate = statistics.median(estimates)
print("FM estimate without grouping:", round(fm_estimate))

In [ ]:
# Test implementation with grouping
groups = [registers[i:i+FM_GROUP_SIZES] for i in range(0, len(registers), FM_GROUP_SIZES)]
group_means = [sum(g)/len(g) for g in groups]
group_estimates = [(2**m)/FM_PHI for m in group_means]
estimate = statistics.median(group_estimates)
print("FM estimate with grouping", round(estimate))

In [ ]:
# For error calculation
actual = df.select("userID").distinct().count()
print("Correct result: ", actual)

# Compare with HLL to see how worse it is
from pyspark.sql.functions import approx_count_distinct
print(f"HyperLogLog estimate {df.select(approx_count_distinct("userID")).collect()}")

# AMS Algorithm Implementation